In [1]:
import numpy as np, sys
print("numpy:", np.__version__)
print("numpy file:", np.__file__)
print("python:", sys.executable)

numpy: 1.26.4
numpy file: /usr/local/share/pynq-venv/lib/python3.10/site-packages/numpy/__init__.py
python: /usr/local/share/pynq-venv/bin/python3


In [2]:
from pynq import Overlay

overlay = Overlay('activation_accelerator.bit')

In [3]:
help(overlay)

Help on Overlay in module pynq.overlay:

<pynq.overlay.Overlay object>
    Default documentation for overlay activation_accelerator.bit. The following
    attributes are available on this overlay:
    
    IP Blocks
    ----------
    accelerator_control_0 : pynq.overlay.DefaultIP
    zynq_ultra_ps_e_0    : pynq.overlay.DefaultIP
    
    Hierarchies
    -----------
    None
    
    Interrupts
    ----------
    None
    
    GPIO Outputs
    ------------
    None
    
    Memories
    ------------
    PSDDR                : Memory



In [4]:
acc_ip = overlay.accelerator_control_0
help(acc_ip)#activation_accelerat_0 accelerator_control_0

Help on DefaultIP in module pynq.overlay object:

class DefaultIP(builtins.object)
 |  DefaultIP(description)
 |  
 |  Driver for an IP without a more specific driver
 |  
 |  This driver wraps an MMIO device and provides a base class
 |  for more specific drivers written later. It also provides
 |  access to GPIO outputs and interrupts inputs via attributes. More specific
 |  drivers should inherit from `DefaultIP` and include a
 |  `bindto` entry containing all of the IP that the driver
 |  should bind to. Subclasses meeting these requirements will
 |  automatically be registered.
 |  
 |  Attributes
 |  ----------
 |  mmio : pynq.MMIO
 |      Underlying MMIO driver for the device
 |  _interrupts : dict
 |      Subset of the PL.interrupt_pins related to this IP
 |  _gpio : dict
 |      Subset of the PL.gpio_dict related to this IP
 |  
 |  Methods defined here:
 |  
 |  __init__(self, description)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  read(se

In [5]:
def calculate_point(error_true):
    ERROR_point= 0
    if error_true <= 1e-3:
        ERROR_point = 1
    elif error_true > 1e-3 and error_true <= 0.1:
        ERROR_point = (np.log(0.1) - np.log(error_true))/np.log(100)
    else:
        ERROR_point = 0
    return ERROR_point

# 计算L2误差
# ε_f = ||Y_fpga,f - Y_ref,f||_2 / (||Y_ref,f||_2 + 10^{-12})

def l2_error(golden, output):
    """
    计算两个数组之间的相对L2误差，增强数值稳定性
    """
    # 使用float64进行计算以避免溢出
    f32_max = np.finfo(np.float32).max
    f32_min = -f32_max

    if output.shape != golden.shape:
        raise ValueError(f"Shape mismatch: {output.shape} vs {golden.shape}")

    # 转换为 float64 便于更稳定的计算
    output_f64 = np.array(output, dtype=np.float64)
    golden_f64 = np.array(golden, dtype=np.float64)

    eps = 1e-12
    BIG_VAL = 1e9  # 减小惩罚值，避免溢出

    # 使用更安全的NaN/Inf处理方式
    output_safe = output_f64.copy()
    golden_safe = golden_f64.copy()

    # 创建掩码
    nan_mask_out = np.isnan(output_safe)
    nan_mask_gold = np.isnan(golden_safe)
    inf_mask_out = np.isinf(output_safe)
    inf_mask_gold = np.isinf(golden_safe)

    # 处理NaN
    both_nan = nan_mask_out & nan_mask_gold
    only_out_nan = nan_mask_out & ~nan_mask_gold
    only_gold_nan = ~nan_mask_out & nan_mask_gold

    output_safe[both_nan] = 0
    golden_safe[both_nan] = 0
    output_safe[only_out_nan] = BIG_VAL
    golden_safe[only_out_nan] = 0
    output_safe[only_gold_nan] = 0
    golden_safe[only_gold_nan] = BIG_VAL

    # 处理Inf
    both_inf = inf_mask_out & inf_mask_gold
    only_out_inf = inf_mask_out & ~inf_mask_gold
    only_gold_inf = ~inf_mask_out & inf_mask_gold

    # 检查Inf符号是否相同
    inf_sign_diff = both_inf & (np.sign(output_safe) != np.sign(golden_safe))
    inf_sign_same = both_inf & (np.sign(output_safe) == np.sign(golden_safe))

    output_safe[inf_sign_same] = 0
    golden_safe[inf_sign_same] = 0
    output_safe[inf_sign_diff] = BIG_VAL
    golden_safe[inf_sign_diff] = 0
    output_safe[only_out_inf] = BIG_VAL
    golden_safe[only_out_inf] = 0
    output_safe[only_gold_inf] = 0
    golden_safe[only_gold_inf] = BIG_VAL

    # 安全的L2范数计算
    with np.errstate(over='ignore', invalid='ignore'):
        # 计算差值
        diff = output_safe - golden_safe

        # 使用更稳定的范数计算方法
        # 避免直接计算大数的平方
        max_abs_val = max(np.max(np.abs(output_safe)), np.max(np.abs(golden_safe)), 1.0)

        # 归一化后再计算范数
        scale_factor = 1.0 / max_abs_val if max_abs_val > 0 else 1.0

        diff_scaled = diff * scale_factor
        golden_scaled = golden_safe * scale_factor

        numerator_all = np.linalg.norm(diff_scaled, ord=2)
        denominator_all = np.linalg.norm(golden_scaled, ord=2) + eps

        # 如果分母仍然为0或极小，使用备选方案
        if denominator_all < eps:
            # 使用绝对误差代替相对误差
            mean_abs_golden = np.mean(np.abs(golden_safe)) + eps
            errors = numerator_all / mean_abs_golden
        else:
            errors = numerator_all / denominator_all

    errors_point = calculate_point(errors)

    return errors, errors_point

In [7]:
from pynq import allocate
import numpy as np
import time, os
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
# ====== 1) 准备数据 ======
in0_path = "X_test_tensor_bf16.bin"
in1_path = "Y_test_tensor_bf16.bin"

# in0_path = "X_test_tensor_bf16_2.bin"
# in1_path = "Y_test_tensor_bf16_2.bin"

# 假设每个 config 有各自的 golden（按需改路径/命名）
golden_paths = {
    0: "ref_softmax_bf16.bin",
    1: "ref_layernorm_bf16.bin",
    2: "ref_rmsnorm_bf16.bin",
    3: "ref_silu_bf16.bin",
    4: "ref_gelu_bf16.bin",
    5: "ref_add_bf16.bin",
    6: "ref_mul_bf16.bin",
}
# golden_paths = {
#     0: "ref_softmax_bf16_2.bin",
#     1: "ref_layernorm_bf16_2.bin",
#     2: "ref_rmsnorm_bf16_2.bin",
#     3: "ref_silu_bf16_2.bin",
#     4: "ref_gelu_bf16_2.bin",
#     5: "ref_add_bf16_2.bin",
#     6: "ref_mul_bf16_2.bin",
# }
# 输入按 bf16（二字节）加载为 uint16 原始位宽
arr0 = np.fromfile(in0_path, dtype=np.uint16)
arr1 = np.fromfile(in1_path, dtype=np.uint16)
print(arr0.shape)
# 申请可 DMA 的连续物理内存
buf0 = allocate(shape=arr0.shape, dtype=np.uint16)
buf1 = allocate(shape=arr1.shape, dtype=np.uint16)
out  = allocate(shape=arr0.shape, dtype=np.uint16)  # 假设输出长度与输入相同

# 拷贝并刷新到 DDR
np.copyto(buf0, arr0); buf0.flush()
np.copyto(buf1, arr1); buf1.flush()
np1 = buf0.copy()
np1.tofile("input0.bin")
np2 = buf1.copy()
np2.tofile("input1.bin")
out[:] = 0; out.flush()

# 物理地址
pa0 = int(buf0.physical_address)
pa1 = int(buf1.physical_address)
pao = int(out.physical_address)
print("pa0 align:", pa0 % 8)
print("pa1 align:", pa1 % 8)
print("pao align:", pao % 8)
# ====== 2) 写地址寄存器（与你 rpt 的偏移一致）======
def wr(off, val): acc_ip.write(off, int(val) & 0xFFFFFFFF)
def rd(off): return acc_ip.read(off)

wr(0x10,  pa0 & 0xFFFFFFFF)       # in0 low
wr(0x14, (pa0 >> 32) & 0xFFFFFFFF)# in0 high
wr(0x1C,  pa1 & 0xFFFFFFFF)       # in1 low
wr(0x20, (pa1 >> 32) & 0xFFFFFFFF)# in1 high
wr(0x28,  pao & 0xFFFFFFFF)       # out low
wr(0x2C, (pao >> 32) & 0xFFFFFFFF)# out high

print("in0 = 0x%08X_%08X"%(rd(0x14), rd(0x10)))
print("in1 = 0x%08X_%08X"%(rd(0x20), rd(0x1C)))
print("out = 0x%08X_%08X"%(rd(0x2C), rd(0x28)))

#====== 3) stage 0：把 in0/in1“加载到 BRAM/片上” ======
wr(0x34, 0)   # stage = 0
wr(0x00, 1)   # ap_start
t0 = time.perf_counter()
while (rd(0x00) & 0x2) == 0:  # 等 ap_done
    if time.perf_counter() - t0 > 5.0:
        raise TimeoutError("stage 0 超时，检查 IP/时钟/复位")
t1 = time.perf_counter()
print(f"stage-0 done; CTRL={hex(rd(0x00))}, TIME={t1-t0:.8f}")

# ====== 4) stage 1：循环跑各个 config，计时 ======
total_time = 0.0
results = {}
def bf16_to_f32(u16_arr: np.ndarray) -> np.ndarray:
    u32 = (u16_arr.astype(np.uint32) << 16)
    return u32.view(np.float32)

for cfg in [0,1,2,3,4,5,6]:
    # 设置 config
    wr(0x3C, cfg)
    wr(0x34, 1)        # stage = 1（计算）
    one_time = 0.0
    wr(0x00, 1)
    for i in range(10000):
        # 启动
        
        # 等待完成
        t_start = time.perf_counter()
        while (rd(0x00) & 0x2) == 0:
            # if time.perf_counter() - t_start > 10.0:
            #     raise TimeoutError(f"stage 1(config={cfg}) 超时")
            pass
        t_end = time.perf_counter()
        one_time += (t_end - t_start)
    total_time += one_time/10000
    print("===============计算时间统计==============")
    print(f"[config={cfg}] compute time = {one_time*100:.8f} us")

    # ====== 5) stage 2：把结果写回 DDR（如果你的 IP 需要）======
    wr(0x34, 2)   # stage = 2（搬运）
    wr(0x00, 1)
    t2 = time.time()
    while (rd(0x00) & 0x2) == 0:
        if time.time() - t2 > 5.0:
            raise TimeoutError(f"stage 2(config={cfg}) 超时")
        time.sleep(0.001)

    # 失效缓存，读取 out
    out.invalidate()
    out_vec = out.copy()  # 保存这一轮的输出
    results[cfg] = out_vec

    # ====== 6) 对比 golden（若有）======
    gpath = golden_paths.get(cfg, None)
    if gpath and os.path.exists(gpath):
        golden = np.fromfile(gpath, dtype=np.uint16)
        if golden.shape != out_vec.shape:
            print(f"[config={cfg}] GOLDEN 形状不一致: golden={golden.shape}, out={out_vec.shape}")
        else:
            output_filename = f"fpga_out{cfg}.bin"
            out_vec.tofile(output_filename)
            print(f"[config={cfg}] 结果已保存到 {output_filename}")
            same = np.array_equal(golden, out_vec)
            diff = np.count_nonzero(golden != out_vec) # 统计不相等元素的个数
            g_f = bf16_to_f32(golden)
            o_f = bf16_to_f32(out_vec)
            abs_diff = np.abs(g_f - o_f)
            dif_l2_error, points = l2_error(g_f, o_f)
            mask = np.isfinite(g_f) & np.isfinite(o_f)
            if mask.any():
                # 为了拿到“原始索引”的最大差，给非有限值设为 -inf 后做 argmax
                finite_abs = abs_diff.copy()
                finite_abs[~mask] = -np.inf
                idx_max = int(np.argmax(finite_abs))
                max_abs_diff = float(abs_diff[idx_max])
            else:
                idx_max = -1
                max_abs_diff = float('nan')
            print("===============计算结果统计==============")
            print(f"[config={cfg}] compare golden: equal(bits)={same}, diff_count={diff}, max_abs_diff={max_abs_diff}")
            print(f"l2_error={dif_l2_error}, points = {points}")
            if idx_max >= 0:
                print(f"  worst@{idx_max}: golden={g_f[idx_max]}, out={o_f[idx_max]}, abs_diff={abs_diff[idx_max]}")
    else:
        print(f"[config={cfg}] 没有提供 golden 文件（跳过对比）")

print(f"Total compute time (stage1 sum) = {total_time:.8f} s")


(49152,)
pa0 align: 0
pa1 align: 0
pao align: 0
in0 = 0x00000000_37C40000
in1 = 0x00000000_375C0000
out = 0x00000000_375E0000
stage-0 done; CTRL=0x3, TIME=0.00048113
===============计算时间统计==============
[config=0] compute time = 9.50552980 us
[config=0] 结果已保存到 fpga_out0.bin
===============计算结果统计==============
[config=0] compare golden: equal(bits)=False, diff_count=9537, max_abs_diff=0.00048828125
l2_error=0.00031877686381148274, points = 1
  worst@49078: golden=0.07666015625, out=0.076171875, abs_diff=0.00048828125
===============计算时间统计==============
[config=1] compute time = 9.60905920 us
[config=1] 结果已保存到 fpga_out1.bin
===============计算结果统计==============
[config=1] compare golden: equal(bits)=False, diff_count=21120, max_abs_diff=2.0594661026406477e-35
l2_error=6.287262948436976e-36, points = 1
  worst@37668: golden=1.2695338988880705e-35, out=3.329000001528718e-35, abs_diff=2.0594661026406477e-35
===============计算时间统计==============
[config=2] compute time = 9.37764750 us
[config=2] 